# Binary PREICTAL vs INTERICTAL Labeling (5 / 15 / 30-Minute Horizons)

**Task scope (per supervisor's methodological request).** Replace the
existing 4-class time-to-seizure forecasting target
(`0-30s` / `30-60s` / `60-120s` / `>120s`, used in
`EEG_TCN_seizure_forecasting.ipynb` and
`EEG_TCN_state_deviation_forecasting.ipynb`) with a **binary**
PREICTAL vs INTERICTAL label, configurable for three preictal horizons:
**5, 15, and 30 minutes**.

**This notebook does ONLY labeling.** It does not train a model, does not
normalize/scale features, does not perform any train/val/test split, and
does not touch the existing 4-class notebooks or their saved outputs. It
reads the already-verified CHB24 + CHB12 qEEG feature tables (read-only)
and writes one new, clearly-named output file.

**Reused, unmodified inputs:**
- `outputs/features_chb24_forecasting.parquet`
- `outputs/features_chb12_forecasting.parquet`

Both already carry, per 30s epoch: the 14 qEEG features, `recording_id`,
`dataset`, `subject_id`, `epoch_idx`, `t_start_sec`/`t_center_sec`/`t_end_sec`,
`y_seizure` (seizure-overlap flag), `next_seizure_start_sec`, and
`time_to_next_seizure_sec` -- all produced by `EEG_CHB24_preprocessing.ipynb`
/ `EEG_CHB12_preprocessing.ipynb` and left untouched here.

**What is genuinely new here:** `time_since_previous_seizure_sec` (and
`seizure_id` / `next_seizure_id` / `previous_seizure_id`) do NOT exist in
either table today -- only "time to the NEXT seizure" was ever computed.
Postictal exclusion (the supervisor's requirement) needs "time SINCE the
PREVIOUS seizure ENDED", which requires the seizure END times. Rather than
approximate seizure end from 30s-epoch-level `y_seizure` flags (rounding to
epoch granularity), this notebook re-derives it from the same authoritative
source the existing tables were built from -- the CHB-MIT `chbXX-summary.txt`
annotation files -- via the exact `parse_chb_summary` parser already
verified in `EEG_CHB24_preprocessing.ipynb` / `EEG_CHB12_preprocessing.ipynb`
(copied verbatim below, credited). A cross-check below confirms the
freshly-parsed intervals reproduce the *existing* `next_seizure_start_sec`
column exactly, before anything new is derived from them.


## 1. Load the existing (unmodified) feature tables

In [1]:

import re
import uuid
import urllib.request
import urllib.error
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 60)
print("BINARY PREICTAL/INTERICTAL LABELING -- SETUP")
print("=" * 60)
print(f"numpy : {np.__version__}")
print(f"pandas: {pd.__version__}")

# Portable path resolution: this notebook is expected to be opened with its
# working directory at the repository root (Jupyter's default when a
# notebook is launched from the repo folder), matching how the rest of the
# project's notebooks are run day-to-day.
repo_root = Path.cwd()
output_dir = repo_root / "outputs"
chb_dir = repo_root / "data" / "raw" / "chbmit"
chb_dir.mkdir(parents=True, exist_ok=True)

chb24_path = output_dir / "features_chb24_forecasting.parquet"
chb12_path = output_dir / "features_chb12_forecasting.parquet"

for p in (chb24_path, chb12_path):
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found. This notebook does not regenerate CHB24/CHB12 "
            f"preprocessing -- run EEG_CHB24_preprocessing.ipynb / "
            f"EEG_CHB12_preprocessing.ipynb first."
        )

df24 = pd.read_parquet(chb24_path)
df12 = pd.read_parquet(chb12_path)

REQUIRED_COLS = {
    "recording_id", "dataset", "subject_id", "epoch_idx",
    "t_start_sec", "t_center_sec", "t_end_sec",
    "y_seizure", "next_seizure_start_sec", "time_to_next_seizure_sec",
}
for name, df in [("CHB24", df24), ("CHB12", df12)]:
    missing = REQUIRED_COLS - set(df.columns)
    if missing:
        raise ValueError(f"{name} table missing required columns: {missing}")
    print(f"{name}: shape={df.shape}, recordings={sorted(df['recording_id'].unique())}, "
          f"subject_id={sorted(df['subject_id'].unique())}")

FEATURE_COLS = [
    "std", "var", "rms", "delta", "theta", "alpha", "beta",
    "rel_delta", "rel_theta", "rel_alpha", "rel_beta",
    "theta_alpha_ratio", "delta_beta_ratio", "entropy",
]
missing_feat = [c for c in FEATURE_COLS if c not in df24.columns or c not in df12.columns]
if missing_feat:
    raise ValueError(f"Expected qEEG feature columns missing: {missing_feat}")

print(f"\n[OK] Both tables loaded read-only. {len(FEATURE_COLS)} qEEG feature "
      f"columns confirmed present in both.")


BINARY PREICTAL/INTERICTAL LABELING -- SETUP
numpy : 2.0.2
pandas: 2.3.3
CHB24: shape=(1440, 24), recordings=['chb_s24_r01', 'chb_s24_r03', 'chb_s24_r04', 'chb_s24_r06', 'chb_s24_r07', 'chb_s24_r09', 'chb_s24_r11', 'chb_s24_r13', 'chb_s24_r14', 'chb_s24_r15', 'chb_s24_r17', 'chb_s24_r21'], subject_id=[np.int64(24)]
CHB12: shape=(600, 24), recordings=['chb_s12_r06', 'chb_s12_r08', 'chb_s12_r23', 'chb_s12_r38', 'chb_s12_r42'], subject_id=[np.int64(12)]

[OK] Both tables loaded read-only. 14 qEEG feature columns confirmed present in both.


## 2. Inspect the current 4-class labeling (as implemented today)

Confirmed by reading `EEG_TCN_seizure_forecasting.ipynb` and
`EEG_TCN_state_deviation_forecasting.ipynb` directly (both construct the
target identically, from the same two metadata columns):

- **`time_to_next_seizure_sec`** is computed once, in each preprocessing
  notebook (`EEG_CHB24_preprocessing.ipynb` Step 8 / `EEG_CHB12_preprocessing.ipynb`
  Step 9), by `compute_time_to_next_seizure()`: for every epoch, find the
  first seizure **onset** in the *same recording* with `onset >= t_end_sec`
  (`np.searchsorted`), and set `time_to_next_seizure_sec = onset - t_end_sec`.
  It is `NaN` when no future seizure remains in that recording. It is
  computed for *every* epoch, including seizure-overlapping ones (their value
  refers to the *next* seizure, not the one they're inside).
- **Seizure-overlapping epochs are excluded from the target**, not
  mislabeled: both notebooks build `forecast_class` only on the
  `y_seizure == 0` subset (`EEG_TCN_seizure_forecasting.ipynb` cell 3;
  `EEG_TCN_state_deviation_forecasting.ipynb` cell 4, which additionally
  assigns them the sentinel `forecast_class = -1`).
- **The 4 classes** are cut directly from `time_to_next_seizure_sec` (`dt`):
  `0<=dt<=30` &rarr; class 0 (`0-30s`), `30<dt<=60` &rarr; class 1
  (`30-60s`), `60<dt<=120` &rarr; class 2 (`60-120s`), everything else
  (`dt>120` or `dt` is `NaN`, i.e. no future seizure at all) &rarr; class 3
  (`>120s`). `dt==0` is intentionally inclusive in class 0 (the epoch's END
  lands exactly on the seizure's onset second).
- **Recordings/patients involved:** CHB24 (single patient, 12
  seizure-containing recordings, `subject_id=24`) in
  `EEG_TCN_seizure_forecasting.ipynb`; CHB24 + CHB12 (`subject_id` 24 and 12,
  17 recordings total -- CHB12 expansion was time-limited to 5 of its 10
  montage-compatible seizure recordings) in
  `EEG_TCN_state_deviation_forecasting.ipynb`.
- **Postictal handling: there is none.** Neither preprocessing notebook nor
  either forecasting notebook computes anything about time *since* a
  *previous* seizure. `>120s`/class 3 (and, in the state-deviation notebook,
  the "stable TRAIN baseline" used to fit the scaler/PCA) is defined purely
  by *distance to the next seizure*; an epoch 10 seconds after a seizure
  ENDED, if it is also >120s before the *next* seizure, is currently treated
  identically to a genuinely quiet interictal epoch. This is exactly the
  methodological gap the supervisor's requirement addresses.
- **What already exists that is reused below:** `y_seizure`,
  `next_seizure_start_sec`, `time_to_next_seizure_sec`, `recording_id`,
  `subject_id`, `epoch_idx`, `t_start_sec`/`t_end_sec`, and all 14 qEEG
  feature columns -- none of this needs to be recomputed.


## 3. Recover seizure END times (needed for postictal exclusion)

`time_since_previous_seizure_sec` is not present in either saved table. It
is derived here from the seizure annotation files themselves
(`chb24-summary.txt` / `chb12-summary.txt`), using `parse_chb_summary`
copied verbatim from `EEG_CHB24_preprocessing.ipynb` /
`EEG_CHB12_preprocessing.ipynb` (the CHB12 variant generalizes the regex to
`Seizure(?:\s+\d+)?\s+Start/End Time:` to also match CHB12's
multi-seizure-per-file numbering, e.g. `Seizure 2 Start Time:`). This is the
same parser that produced the `y_seizure` / `next_seizure_start_sec` columns
already in the saved tables -- reusing it (rather than re-deriving seizure
timing some other way) is what keeps the new metadata *consistent* with the
existing metadata. That consistency is checked explicitly below: the
onsets recovered here must reproduce the existing `next_seizure_start_sec`
column exactly.


In [2]:

def download_if_missing(url, dest_path):
    '''Idempotent download (verbatim convention from the preprocessing notebooks).'''
    if dest_path.exists():
        print(f"  [SKIP]      {dest_path.name} already exists")
        return
    try:
        print(f"  [DOWNLOAD]  {dest_path.name} <- {url}")
        urllib.request.urlretrieve(url, dest_path)
        print(f"  [OK]        {dest_path.name} ({dest_path.stat().st_size/1e3:.1f} KB)")
    except (urllib.error.URLError, urllib.error.HTTPError) as e:
        if dest_path.exists():
            dest_path.unlink()
        raise RuntimeError(f"FAILED to download {url}: {e}") from e


def parse_chb_summary(summary_path, known_edf_filenames):
    '''
    Parse a CHB-MIT *-summary.txt file into {edf_filename: [(start_s, end_s), ...]}.

    Copied verbatim (parsing logic) from EEG_CHB24_preprocessing.ipynb /
    EEG_CHB12_preprocessing.ipynb, using the CHB12 notebook's generalized
    regex ("Seizure(?:\\s+\\d+)?\\s+Start/End Time:") so a single function
    handles both the single-seizure-per-file (CHB24) and multi-seizure-per-
    file (CHB12) summary formats.
    '''
    text = Path(summary_path).read_text()
    blocks = re.split(r"(?=File Name: )", text)

    annotations = {}
    for block in blocks:
        name_match = re.search(r"File Name:\s*(\S+)", block)
        if not name_match:
            continue
        fname = name_match.group(1)
        if fname not in known_edf_filenames:
            continue

        n_match = re.search(r"Number of Seizures in File:\s*(\d+)", block)
        if not n_match:
            raise ValueError(f"{fname}: could not find seizure count in summary")
        n_seizures = int(n_match.group(1))

        starts = [int(s) for s in re.findall(r"Seizure(?:\s+\d+)?\s+Start Time:\s*(\d+)\s*seconds", block)]
        ends = [int(s) for s in re.findall(r"Seizure(?:\s+\d+)?\s+End Time:\s*(\d+)\s*seconds", block)]

        if len(starts) != n_seizures or len(ends) != n_seizures:
            raise ValueError(
                f"{fname}: declared {n_seizures} seizures but parsed "
                f"{len(starts)} start / {len(ends)} end times"
            )
        annotations[fname] = list(zip(starts, ends))

    missing = set(known_edf_filenames) - set(annotations)
    if missing:
        raise ValueError(f"No summary entry found for: {missing}")
    return annotations


def recording_id_to_edf_filename(recording_id, subject_id):
    ''''chb_s24_r01', subject_id=24 -> 'chb24_01.edf' (project's established naming).'''
    r_part = recording_id.split("_")[-1]
    assert r_part.startswith("r"), recording_id
    return f"chb{int(subject_id):02d}_{r_part[1:]}.edf"


def load_seizure_intervals(df, summary_path):
    subject_id = int(df["subject_id"].iloc[0])
    recording_ids = sorted(df["recording_id"].unique())
    edf_filenames = {rid: recording_id_to_edf_filename(rid, subject_id) for rid in recording_ids}
    by_file = parse_chb_summary(summary_path, known_edf_filenames=set(edf_filenames.values()))
    return {rid: sorted(by_file[fname]) for rid, fname in edf_filenames.items()}


chb24_summary_path = chb_dir / "chb24-summary.txt"
chb12_summary_path = chb_dir / "chb12-summary.txt"
download_if_missing("https://physionet.org/files/chbmit/1.0.0/chb24/chb24-summary.txt", chb24_summary_path)
download_if_missing("https://physionet.org/files/chbmit/1.0.0/chb12/chb12-summary.txt", chb12_summary_path)

intervals24 = load_seizure_intervals(df24, chb24_summary_path)
intervals12 = load_seizure_intervals(df12, chb12_summary_path)

print("\nParsed seizure intervals (onset_sec, end_sec), recording-relative:")
for rid, iv in sorted({**intervals24, **intervals12}.items()):
    print(f"  {rid}: {iv}")

n_seizures_24 = sum(len(v) for v in intervals24.values())
n_seizures_12 = sum(len(v) for v in intervals12.values())
print(f"\nTotal seizures: CHB24={n_seizures_24}, CHB12={n_seizures_12}, combined={n_seizures_24+n_seizures_12}")


  [DOWNLOAD]  chb24-summary.txt <- https://physionet.org/files/chbmit/1.0.0/chb24/chb24-summary.txt
  [OK]        chb24-summary.txt (2.2 KB)
  [DOWNLOAD]  chb12-summary.txt <- https://physionet.org/files/chbmit/1.0.0/chb12/chb12-summary.txt
  [OK]        chb12-summary.txt (7.4 KB)

Parsed seizure intervals (onset_sec, end_sec), recording-relative:
  chb_s12_r06: [(1665, 1726), (3415, 3447)]
  chb_s12_r08: [(1426, 1439), (1591, 1614), (1957, 1977), (2798, 2824)]
  chb_s12_r23: [(253, 333), (425, 522), (630, 670)]
  chb_s12_r38: [(1548, 1573), (2798, 2821), (2966, 3009), (3146, 3201), (3364, 3410)]
  chb_s12_r42: [(699, 750), (945, 973), (1170, 1199), (1676, 1701), (2213, 2236)]
  chb_s24_r01: [(480, 505), (2451, 2476)]
  chb_s24_r03: [(231, 260), (2883, 2908)]
  chb_s24_r04: [(1088, 1120), (1411, 1438), (1745, 1764)]
  chb_s24_r06: [(1229, 1253)]
  chb_s24_r07: [(38, 60)]
  chb_s24_r09: [(1745, 1764)]
  chb_s24_r11: [(3527, 3597)]
  chb_s24_r13: [(3288, 3304)]
  chb_s24_r14: [(1939, 196

In [3]:

def cross_check_against_existing_next_seizure(df, intervals_by_recording, label):
    '''The freshly re-parsed onsets must exactly reproduce the EXISTING
    next_seizure_start_sec column already saved in the parquet table --
    i.e. the new metadata is derived from the identical source of truth,
    not a second, potentially inconsistent, re-derivation.'''
    recomputed = np.full(len(df), np.nan)
    for recording_id, intervals in intervals_by_recording.items():
        mask = (df["recording_id"] == recording_id).to_numpy()
        onsets = np.sort(np.array([s for s, _ in intervals], dtype=float))
        t_end = df.loc[mask, "t_end_sec"].to_numpy()
        idx = np.searchsorted(onsets, t_end, side="left")
        next_onset = np.where(idx < len(onsets), onsets[np.clip(idx, 0, len(onsets) - 1)], np.nan)
        recomputed[mask] = next_onset
    existing = df["next_seizure_start_sec"].to_numpy()
    both_nan = np.isnan(existing) & np.isnan(recomputed)
    close = np.isclose(existing, recomputed, equal_nan=False)
    if not (both_nan | close).all():
        bad = (~(both_nan | close)).sum()
        raise ValueError(f"{label}: cross-check FAILED, {bad} rows disagree with existing next_seizure_start_sec")
    print(f"[OK] {label}: freshly parsed seizure intervals reproduce the existing "
          f"next_seizure_start_sec column exactly ({len(df)} rows checked).")

cross_check_against_existing_next_seizure(df24, intervals24, "CHB24")
cross_check_against_existing_next_seizure(df12, intervals12, "CHB12")


[OK] CHB24: freshly parsed seizure intervals reproduce the existing next_seizure_start_sec column exactly (1440 rows checked).
[OK] CHB12: freshly parsed seizure intervals reproduce the existing next_seizure_start_sec column exactly (600 rows checked).


## 4. Add previous-seizure adjacency metadata

New, purely additive columns (existing columns are never modified):

| column | meaning |
|---|---|
| `seizure_id` | identity of the seizure THIS epoch overlaps (`y_seizure==1` rows only), else `None` |
| `next_seizure_id` | identity of the seizure `next_seizure_start_sec` refers to |
| `previous_seizure_end_sec` | END time (recording-relative seconds) of the most recent PAST seizure whose end `<= t_start_sec` |
| `previous_seizure_id` | identity of that seizure |
| `time_since_previous_seizure_sec` | `t_start_sec - previous_seizure_end_sec` (`NaN` if no earlier seizure in the recording) |

`seizure_id` format: `"<recording_id>_sz<k>"`, `k` = 0-indexed temporal
order of that seizure within its recording (e.g. `chb_s24_r04_sz1` is the
second of CHB24 recording r04's three seizures).


In [4]:

def add_seizure_adjacency_metadata(df, intervals_by_recording):
    df = df.copy()
    df["seizure_id"] = None
    df["next_seizure_id"] = None
    df["previous_seizure_end_sec"] = np.nan
    df["previous_seizure_id"] = None
    df["time_since_previous_seizure_sec"] = np.nan

    for recording_id, intervals in intervals_by_recording.items():
        mask = (df["recording_id"] == recording_id).to_numpy()
        if not mask.any():
            continue
        idx_rows = df.index[mask]

        onsets = np.array([s for s, _ in intervals], dtype=float)
        ends = np.array([e for _, e in intervals], dtype=float)
        seiz_ids = [f"{recording_id}_sz{k}" for k in range(len(intervals))]

        t_start = df.loc[idx_rows, "t_start_sec"].to_numpy()
        t_end = df.loc[idx_rows, "t_end_sec"].to_numpy()

        # seizure_id: which annotated interval (if any) this epoch overlaps.
        seizure_id_col = np.full(len(idx_rows), None, dtype=object)
        for k, (onset, end) in enumerate(intervals):
            overlap = (t_start < end) & (t_end > onset)
            for pos in np.where(overlap)[0]:
                seizure_id_col[pos] = seiz_ids[k]
        df.loc[idx_rows, "seizure_id"] = seizure_id_col

        # next_seizure_id: identical rule to the existing next_seizure_start_sec
        # (first onset >= t_end) -- just also records WHICH seizure that is.
        next_idx = np.searchsorted(onsets, t_end, side="left")
        next_id_col = np.array([seiz_ids[i] if i < len(onsets) else None for i in next_idx], dtype=object)
        df.loc[idx_rows, "next_seizure_id"] = next_id_col

        # previous_seizure_end_sec: last END <= t_start (most recent PAST seizure).
        order = np.argsort(ends)
        ends_sorted = ends[order]
        ids_sorted = [seiz_ids[i] for i in order]
        prev_idx = np.searchsorted(ends_sorted, t_start, side="right") - 1
        prev_end_col = np.where(prev_idx >= 0, ends_sorted[np.clip(prev_idx, 0, None)], np.nan)
        prev_id_col = np.array([ids_sorted[i] if i >= 0 else None for i in prev_idx], dtype=object)
        df.loc[idx_rows, "previous_seizure_end_sec"] = prev_end_col
        df.loc[idx_rows, "previous_seizure_id"] = prev_id_col

    valid_prev = df["previous_seizure_end_sec"].notna()
    df.loc[valid_prev, "time_since_previous_seizure_sec"] = (
        df.loc[valid_prev, "t_start_sec"] - df.loc[valid_prev, "previous_seizure_end_sec"]
    )

    finite_tsp = df["time_since_previous_seizure_sec"].dropna()
    assert (finite_tsp >= 0).all(), "time_since_previous_seizure_sec must be non-negative"
    assert (df["time_since_previous_seizure_sec"].isna() == df["previous_seizure_end_sec"].isna()).all()

    return df


df24_meta = add_seizure_adjacency_metadata(df24, intervals24)
df12_meta = add_seizure_adjacency_metadata(df12, intervals12)

print("[OK] Seizure adjacency metadata added to both tables.")
print("\nExample rows (CHB24, around a seizure boundary):")
example_cols = ["recording_id", "epoch_idx", "y_seizure", "time_to_next_seizure_sec",
                "next_seizure_id", "time_since_previous_seizure_sec", "previous_seizure_id"]
example = df24_meta[(df24_meta["recording_id"] == "chb_s24_r04")].sort_values("epoch_idx")
print(example[example_cols].iloc[30:45].to_string(index=False))

combined = pd.concat([df24_meta, df12_meta], ignore_index=True)
print(f"\nCombined table (CHB24 + CHB12): shape={combined.shape}")


[OK] Seizure adjacency metadata added to both tables.

Example rows (CHB24, around a seizure boundary):
recording_id  epoch_idx  y_seizure  time_to_next_seizure_sec next_seizure_id  time_since_previous_seizure_sec previous_seizure_id
 chb_s24_r04         30          0                     158.0 chb_s24_r04_sz0                              NaN                None
 chb_s24_r04         31          0                     128.0 chb_s24_r04_sz0                              NaN                None
 chb_s24_r04         32          0                      98.0 chb_s24_r04_sz0                              NaN                None
 chb_s24_r04         33          0                      68.0 chb_s24_r04_sz0                              NaN                None
 chb_s24_r04         34          0                      38.0 chb_s24_r04_sz0                              NaN                None
 chb_s24_r04         35          0                       8.0 chb_s24_r04_sz0                              NaN       

## 5. Configurable binary labeling function

```python
make_binary_labels(df, preictal_minutes=5, exclusion_minutes=30)
```

**PREICTAL**: `y_seizure == 0` AND `0 < time_to_next_seizure_sec <= preictal_minutes*60`
AND not postictal-contaminated (see below).

**INTERICTAL**: `y_seizure == 0` AND at least `exclusion_minutes` (>=30, the
supervisor's minimum) away from the next seizure (or no next seizure at
all) AND at least `exclusion_minutes` away from the end of the previous
seizure (or no previous seizure at all).

**Everything else is EXCLUDED** (never forced into a class), tagged with an
explicit `exclusion_reason`:

- `seizure_overlap` -- `y_seizure == 1`.
- `postictal_exclusion` -- within `exclusion_minutes` of the END of the
  previous seizure. This is the supervisor's core requirement: it rejects
  both an otherwise-INTERICTAL epoch that is really postictal, *and* an
  otherwise-PREICTAL epoch whose apparent closeness to the *next* seizure is
  confounded by residual postictal EEG changes from the seizure just
  before it (this matters for clustered seizures -- see the per-seizure
  counts below, where a seizure's own preictal window is often entirely
  consumed by the previous seizure's postictal period).
- `ambiguous_preictal_gap` -- `preictal_minutes < time_to_next_seizure_sec
  <= exclusion_minutes`: too far from the next seizure to count as PREICTAL
  at this horizon, but too close to count as INTERICTAL. Only occurs when
  `preictal_minutes < exclusion_minutes` (i.e. for H=5 and H=15; at H=30
  the two boundaries coincide and this category is empty).
- `seizure_onset_boundary` -- the rare `time_to_next_seizure_sec == 0` case
  (an epoch's END lands exactly on the next seizure's onset second). The
  supervisor's PREICTAL definition is a strict `0 < time_to_next_seizure`,
  so this single-instant boundary is excluded rather than forced into
  PREICTAL (the old 4-class scheme instead included it via an inclusive
  `dt>=0` lower bound -- a deliberate difference from the new binary
  definition, not an oversight).

`exclusion_minutes` is asserted `>= 30` (the supervisor's stated minimum)
and `>= preictal_minutes` (INTERICTAL must be at least as far from a
seizure as the PREICTAL horizon it's being contrasted with) -- both are
hard requirements, not silently clamped.


In [5]:

def make_binary_labels(df, preictal_minutes=5, exclusion_minutes=30):
    if "time_since_previous_seizure_sec" not in df.columns:
        raise ValueError("Run add_seizure_adjacency_metadata() first.")
    if exclusion_minutes < 30:
        raise ValueError("Supervisor requirement: exclusion_minutes must be >= 30.")
    if preictal_minutes > exclusion_minutes:
        raise ValueError("preictal_minutes must be <= exclusion_minutes.")

    preictal_sec = preictal_minutes * 60.0
    exclusion_sec = exclusion_minutes * 60.0

    y_seizure = df["y_seizure"].to_numpy().astype(bool)
    dt_next = df["time_to_next_seizure_sec"].to_numpy()
    dt_prev = df["time_since_previous_seizure_sec"].to_numpy()

    is_near_future = (~np.isnan(dt_next)) & (dt_next <= exclusion_sec)
    is_near_past = (~np.isnan(dt_prev)) & (dt_prev <= exclusion_sec)
    is_preictal_horizon = (~np.isnan(dt_next)) & (dt_next > 0) & (dt_next <= preictal_sec)

    preictal_mask = (~y_seizure) & is_preictal_horizon & (~is_near_past)
    interictal_mask = (~y_seizure) & (~is_near_future) & (~is_near_past)

    label = np.full(len(df), None, dtype=object)
    reason = np.full(len(df), None, dtype=object)
    label[preictal_mask] = "PREICTAL"
    label[interictal_mask] = "INTERICTAL"

    excluded = ~(preictal_mask | interictal_mask)
    reason[excluded & y_seizure] = "seizure_overlap"
    remaining = excluded & (~y_seizure) & (reason == None)
    reason[remaining & is_near_past] = "postictal_exclusion"
    remaining = excluded & (~y_seizure) & (reason == None)
    ambiguous_gap = remaining & (~np.isnan(dt_next)) & (dt_next > preictal_sec) & (dt_next <= exclusion_sec)
    reason[ambiguous_gap] = "ambiguous_preictal_gap"
    remaining = excluded & (~y_seizure) & (reason == None)
    onset_boundary = remaining & (~np.isnan(dt_next)) & (dt_next == 0)
    reason[onset_boundary] = "seizure_onset_boundary"
    remaining = excluded & (~y_seizure) & (reason == None)
    if remaining.any():
        raise AssertionError(f"{remaining.sum()} epochs fell through labeling logic unexplained")

    return label, reason


HORIZONS = [5, 15, 30]
EXCLUSION_MINUTES = 30

for H in HORIZONS:
    label, reason = make_binary_labels(combined, preictal_minutes=H, exclusion_minutes=EXCLUSION_MINUTES)
    combined[f"binary_label_h{H}"] = label
    combined[f"exclusion_reason_h{H}"] = reason

print(f"[OK] Binary labels computed for horizons {HORIZONS} (exclusion_minutes={EXCLUSION_MINUTES}), "
      f"as columns binary_label_h5/h15/h30 on the SAME combined table (no duplicated feature tables).")


[OK] Binary labels computed for horizons [5, 15, 30] (exclusion_minutes=30), as columns binary_label_h5/h15/h30 on the SAME combined table (no duplicated feature tables).


## 6. Validate the labels

Programmatic checks, run for every horizon:
1. No `PREICTAL`/`INTERICTAL` epoch is seizure-overlapping.
2. Every `PREICTAL` epoch's `time_to_next_seizure_sec` is inside
   `(0, H*60]`.
3. Every `INTERICTAL` epoch is `> exclusion_minutes*60` from both the next
   seizure and the end of the previous seizure.
4. No `PREICTAL` epoch is postictal-contaminated (within
   `exclusion_minutes` of a previous seizure's end).
5. Chronological/identity metadata (`recording_id`, `epoch_idx`,
   `t_start_sec`, `t_end_sec`, `subject_id`) has no nulls.

Any failure raises immediately rather than continuing silently.


In [6]:

def validate_binary_labels(df, H, exclusion_minutes=30):
    preictal_sec, exclusion_sec = H * 60.0, exclusion_minutes * 60.0
    lab_col, reason_col = f"binary_label_h{H}", f"exclusion_reason_h{H}"

    pre = df[df[lab_col] == "PREICTAL"]
    inter = df[df[lab_col] == "INTERICTAL"]

    assert (pre["y_seizure"] == 0).all(), "PREICTAL epoch is seizure-overlapping"
    assert (inter["y_seizure"] == 0).all(), "INTERICTAL epoch is seizure-overlapping"

    assert ((pre["time_to_next_seizure_sec"] > 0) & (pre["time_to_next_seizure_sec"] <= preictal_sec)).all(), \
        "A PREICTAL epoch falls outside its horizon"

    inter_next_ok = inter["time_to_next_seizure_sec"].isna() | (inter["time_to_next_seizure_sec"] > exclusion_sec)
    inter_prev_ok = inter["time_since_previous_seizure_sec"].isna() | (inter["time_since_previous_seizure_sec"] > exclusion_sec)
    assert inter_next_ok.all(), "An INTERICTAL epoch is too close to a FUTURE seizure"
    assert inter_prev_ok.all(), "An INTERICTAL epoch is postictal (too close to a PAST seizure)"

    pre_prev_ok = pre["time_since_previous_seizure_sec"].isna() | (pre["time_since_previous_seizure_sec"] > exclusion_sec)
    assert pre_prev_ok.all(), "A PREICTAL epoch is postictal-contaminated"

    for col in ["recording_id", "epoch_idx", "t_start_sec", "t_end_sec", "subject_id"]:
        assert df[col].notna().all(), f"Metadata column {col} has nulls"

    print(f"  [OK] H={H:2d}min: all validation checks passed "
          f"(PREICTAL={len(pre)}, INTERICTAL={len(inter)})")


print("Running validation for every horizon:")
for H in HORIZONS:
    validate_binary_labels(combined, H, EXCLUSION_MINUTES)


Running validation for every horizon:
  [OK] H= 5min: all validation checks passed (PREICTAL=170, INTERICTAL=372)
  [OK] H=15min: all validation checks passed (PREICTAL=445, INTERICTAL=372)
  [OK] H=30min: all validation checks passed (PREICTAL=732, INTERICTAL=372)


## 7. Label counts per horizon

In [7]:

for H in HORIZONS:
    lab_col, reason_col = f"binary_label_h{H}", f"exclusion_reason_h{H}"
    print("=" * 60)
    print(f"H = {H} minutes  (exclusion_minutes = {EXCLUSION_MINUTES})")
    print("=" * 60)

    counts = combined[lab_col].value_counts(dropna=False)
    n_preictal = int(counts.get("PREICTAL", 0))
    n_interictal = int(counts.get("INTERICTAL", 0))
    n_excluded = int(combined[lab_col].isna().sum())
    print(f"PREICTAL   : {n_preictal}")
    print(f"INTERICTAL : {n_interictal}")
    print(f"EXCLUDED   : {n_excluded}")

    print("\nExcluded-epoch breakdown:")
    print(combined.loc[combined[lab_col].isna(), reason_col].value_counts().to_string())

    print("\nPer-patient (subject_id) counts:")
    print(combined.groupby(["subject_id", lab_col], dropna=False).size().unstack(fill_value=0).to_string())

    n_seizures_total = combined["seizure_id"].dropna().apply(lambda s: s).nunique() if False else None
    preictal_rows = combined[combined[lab_col] == "PREICTAL"]
    n_seizures_repr = preictal_rows["next_seizure_id"].nunique()
    print(f"\nSeizures represented by >=1 PREICTAL epoch: {n_seizures_repr}")
    print("\nPer-seizure PREICTAL epoch counts:")
    print(preictal_rows.groupby("next_seizure_id").size().to_string())
    print()


H = 5 minutes  (exclusion_minutes = 30)
PREICTAL   : 170
INTERICTAL : 372
EXCLUDED   : 1498

Excluded-epoch breakdown:
exclusion_reason_h5
postictal_exclusion       866
ambiguous_preictal_gap    562
seizure_overlap            69
seizure_onset_boundary      1

Per-patient (subject_id) counts:
binary_label_h5  INTERICTAL  PREICTAL  NaN
subject_id                                
12                       37        48  515
24                      335       122  983

Seizures represented by >=1 PREICTAL epoch: 19

Per-seizure PREICTAL epoch counts:
next_seizure_id
chb_s12_r06_sz0    10
chb_s12_r08_sz0    10
chb_s12_r23_sz0     8
chb_s12_r38_sz0    10
chb_s12_r42_sz0    10
chb_s24_r01_sz0    10
chb_s24_r01_sz1     4
chb_s24_r03_sz0     7
chb_s24_r03_sz1    10
chb_s24_r04_sz0    10
chb_s24_r06_sz0    10
chb_s24_r07_sz0     1
chb_s24_r09_sz0    10
chb_s24_r11_sz0    10
chb_s24_r13_sz0    10
chb_s24_r14_sz0    10
chb_s24_r15_sz0    10
chb_s24_r17_sz0    10
chb_s24_r21_sz0    10

H = 15 minutes  

## 8. Save new, clearly-named output (nothing existing is overwritten)

Written: `outputs/labels_binary_preictal.parquet` / `.csv` -- ONE combined
CHB24+CHB12 table (not three duplicated datasets) carrying every metadata
column needed for later leave-one-seizure-out modeling, plus a
`binary_label_h{5,15,30}` / `exclusion_reason_h{5,15,30}` column pair per
horizon. The original `features_chb24_forecasting.*` /
`features_chb12_forecasting.*` files, and every existing `outputs/modeling/`
artifact, are untouched.


In [8]:

PRESERVE_COLS = (
    ["patient_id", "subject_id", "recording_id", "dataset", "epoch_idx",
     "t_start_sec", "t_center_sec", "t_end_sec",
     "y_seizure", "seizure_id",
     "next_seizure_id", "next_seizure_start_sec", "time_to_next_seizure_sec",
     "previous_seizure_id", "previous_seizure_end_sec", "time_since_previous_seizure_sec"]
    + FEATURE_COLS
    + [f"binary_label_h{H}" for H in HORIZONS]
    + [f"exclusion_reason_h{H}" for H in HORIZONS]
)

combined["patient_id"] = combined["subject_id"].apply(lambda s: f"chb{int(s):02d}")
final = combined[PRESERVE_COLS].copy()

missing_final = [c for c in PRESERVE_COLS if c not in final.columns]
assert not missing_final, f"Missing expected output columns: {missing_final}"

out_parquet = output_dir / "labels_binary_preictal.parquet"
out_csv = output_dir / "labels_binary_preictal.csv"
final.to_parquet(out_parquet, index=False)
final.to_csv(out_csv, index=False)

print(f"[OK] Saved {final.shape[0]} rows x {final.shape[1]} columns")
print(f"  -> {out_parquet}")
print(f"  -> {out_csv}")
print(f"\nColumns: {list(final.columns)}")

import os
assert os.path.exists(output_dir / "features_chb24_forecasting.parquet")
assert os.path.exists(output_dir / "features_chb12_forecasting.parquet")
print("\n[OK] Existing features_chb24_forecasting.* / features_chb12_forecasting.* untouched.")


[OK] Saved 2040 rows x 36 columns
  -> C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\labels_binary_preictal.parquet
  -> C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\labels_binary_preictal.csv

Columns: ['patient_id', 'subject_id', 'recording_id', 'dataset', 'epoch_idx', 't_start_sec', 't_center_sec', 't_end_sec', 'y_seizure', 'seizure_id', 'next_seizure_id', 'next_seizure_start_sec', 'time_to_next_seizure_sec', 'previous_seizure_id', 'previous_seizure_end_sec', 'time_since_previous_seizure_sec', 'std', 'var', 'rms', 'delta', 'theta', 'alpha', 'beta', 'rel_delta', 'rel_theta', 'rel_alpha', 'rel_beta', 'theta_alpha_ratio', 'delta_beta_ratio', 'entropy', 'binary_label_h5', 'binary_label_h15', 'binary_label_h30', 'exclusion_reason_h5', 'exclusion_reason_h15', 'exclusion_reason_h30']

[OK] Existing features_chb24_forecasting.* / features_chb12_forecasting.* untouched.


## 9. Summary

The old 4-class `forecast_class` target (`EEG_TCN_seizure_forecasting.ipynb`,
`EEG_TCN_state_deviation_forecasting.ipynb`) and its saved model outputs
under `outputs/modeling/tcn_state_deviation_forecasting/` are **untouched**
and remain fully reproducible for project-history/comparison purposes.

This notebook adds, additively:
- `time_since_previous_seizure_sec`, `seizure_id`, `next_seizure_id`,
  `previous_seizure_id` -- derived from the authoritative CHB-MIT summary
  annotations, cross-checked against the existing `next_seizure_start_sec`
  column.
- `make_binary_labels(df, preictal_minutes, exclusion_minutes=30)`, a single
  reusable function driving all three PREICTAL horizons (5/15/30 min) from
  one combined CHB24+CHB12 table.
- `outputs/labels_binary_preictal.parquet` / `.csv`, a new, additively-built
  output file preserving every identifier (`patient_id`, `subject_id`,
  `recording_id`, `seizure_id`/`next_seizure_id`/`previous_seizure_id`,
  `epoch_idx`, `t_start_sec`/`t_end_sec`) needed for a future
  leave-one-seizure-out evaluation.

**Not done in this notebook** (explicitly out of scope per the task):
patient-specific normalization, any new model or TCN redesign, new metrics,
channel changes, multi-patient expansion beyond the existing CHB24+CHB12
data, and any train/val/test split (global scaling and splitting remain the
responsibility of a future modeling notebook, exactly as for the existing
4-class pipeline).
